# Data Preparation Pipeline

Loads EMS call data and CalEnviroScreen hazard data, spatially joins to census tracts, and produces call metrics by 3-month periods (mean & std of calls/week per tract).

In [1]:
import clean

## Full pipeline (one call)

In [2]:
df = clean.prepare_analysis_data(
    ems_path='Fire_Department_and_Emergency_Medical_Services_Dispatched_Calls_for_Service.csv',
    hazards_path='calenviroscreen40resultsdatadictionary_F_2021.xlsx',
)

/Users/evie/Berkeley/STAT230A/project/health/clean.py:36: DtypeWarning: Columns (0: Box) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Standardized columns: ['ozone_std', 'pm2.5_std', 'diesel_pm_std', 'drinking_water_std', 'lead_std', 'pesticides_std', 'tox._release_std', 'traffic_std', 'cleanup_sites_std', 'groundwater_threats_std', 'haz._waste_std', 'imp._water_bodies_std', 'solid_waste_std', 'asthma_std', 'low_birth_weight_std', 'cardiovascular_disease_std', 'education_std', 'linguistic_isolation_std', 'poverty_std', 'unemployment_std', 'housing_burden_std']


In [3]:
print(f"Rows (census tracts): {df.shape[0]}, Columns: {df.shape[1]}")

Rows (census tracts): 182, Columns: 87


In [4]:
call_cols = [c for c in df.columns if 'call_' in c]
print(f"Call metric columns: {call_cols}")

Call metric columns: ['call_avg_aug_oct', 'call_avg_feb_apr', 'call_avg_may_jul', 'call_avg_nov_jan', 'call_std_aug_oct', 'call_std_feb_apr', 'call_std_may_jul', 'call_std_nov_jan']


In [5]:
df[clean.HAZARD_COVARIATES + call_cols].describe()

,ozone,pm2.5,diesel_pm,drinking_water,lead,pesticides,tox._release,traffic,cleanup_sites,groundwater_threats,...,unemployment,housing_burden,call_avg_aug_oct,call_avg_feb_apr,call_avg_may_jul,call_avg_nov_jan,call_std_aug_oct,call_std_feb_apr,call_std_may_jul,call_std_nov_jan
count,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,...,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000
mean,0.031054,8.623882,0.726596,260.157563,58.828028,0.009934,350.977366,1040.861682,8.222527,9.593956,...,4.191209,15.592308,23.464835,22.888791,23.110989,23.815055,8.470495,9.269451,8.351429,8.217637
std,0.001300,0.066518,0.694990,10.785832,14.029507,0.034081,50.752971,627.605542,15.244609,19.090027,...,2.514737,6.628387,31.159210,30.627419,31.154507,30.754050,7.416087,9.414740,7.559536,6.328310
min,0.029372,8.445628,0.079785,258.789440,6.432362,0.000000,221.798665,158.686585,0.000000,0.000000,...,0.000000,4.500000,2.640000,2.400000,2.360000,2.170000,1.800000,1.780000,0.970000,1.700000
25%,0.029372,8.585158,0.240545,258.789440,53.493155,0.000000,314.711669,561.393627,0.000000,0.300000,...,2.600000,11.225000,9.235000,9.652500,9.185000,10.000000,4.510000,5.060000,4.625000,4.820000
50%,0.031908,8.616646,0.475722,258.789440,60.074576,0.000000,350.686831,804.165108,1.750000,3.750000,...,3.750000,14.300000,14.250000,13.790000,13.500000,14.395000,6.220000,6.485000,6.150000,6.565000
75%,0.031908,8.660773,0.989824,258.789440,66.666802,0.000000,387.638250,1415.432537,9.000000,13.225000,...,5.400000,18.550000,20.570000,20.127500,20.570000,21.270000,9.287500,9.130000,8.622500,8.420000
max,0.034190,8.797361,4.751602,363.855387,91.056201,0.272686,499.944661,3273.393341,113.400000,151.800000,...,18.100000,39.000000,247.710000,247.000000,264.290000,248.140000,58.310000,78.800000,65.880000,46.650000


In [6]:
df[['census_tract'] + call_cols + clean.HAZARD_COVARIATES].head()

,census_tract,call_avg_aug_oct,call_avg_feb_apr,call_avg_may_jul,call_avg_nov_jan,call_std_aug_oct,call_std_feb_apr,call_std_may_jul,call_std_nov_jan,ozone,...,imp._water_bodies,solid_waste,asthma,low_birth_weight,cardiovascular_disease,education,linguistic_isolation,poverty,unemployment,housing_burden
0,6075023200,27.00,29.14,20.36,27.07,12.84,13.14,9.36,9.87,0.031908,...,11,20.75,123.98,8.09,11.96,25.3,10.4,27.3,6.1,37.0
1,6075023103,15.93,18.07,16.36,18.79,3.45,9.71,6.10,7.30,0.030640,...,14,13.90,123.98,7.74,11.96,21.0,4.5,71.7,6.4,22.3
2,6075023001,15.21,13.79,11.21,17.50,5.66,7.35,6.45,10.62,0.031908,...,10,5.75,123.98,6.41,11.96,27.5,19.8,30.7,4.8,25.6
3,6075023400,20.14,18.57,20.64,20.93,7.58,7.61,8.79,7.81,0.031908,...,11,16.90,123.98,6.29,11.96,29.7,23.1,40.1,6.3,10.4
4,6075023102,25.00,17.86,24.86,26.64,9.91,7.90,7.12,8.12,0.031908,...,14,9.30,123.98,8.70,11.96,17.8,5.5,48.1,11.6,25.5


## Step-by-step breakdown

In [7]:
hazards, demographics, dictionary = clean.load_hazards_data(
    'calenviroscreen40resultsdatadictionary_F_2021.xlsx'
)
hazards_sf = clean.filter_san_francisco(hazards)
print(f"SF census tracts in hazards data: {len(hazards_sf)}")

SF census tracts in hazards data: 195


In [8]:
ems = clean.load_ems_data(
    'Fire_Department_and_Emergency_Medical_Services_Dispatched_Calls_for_Service.csv'
)
med_inc = clean.filter_medical_incidents(ems)
print(f"Total EMS calls: {len(ems)}, Medical incidents: {len(med_inc)}")

/Users/evie/Berkeley/STAT230A/project/health/clean.py:36: DtypeWarning: Columns (0: Box) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Total EMS calls: 365208, Medical incidents: 246884


In [9]:
med_inc = clean.extract_coords(med_inc, location_col='case_location')
med_inc = clean.assign_nearest_tract(med_inc, hazards_sf)
med_inc = clean.assign_period(med_inc)
call_metrics = clean.aggregate_calls_by_period(med_inc)
print(f"Tracts with call metrics: {len(call_metrics)}")
call_metrics.head()

Tracts with call metrics: 194


,census_tract,call_avg_aug_oct,call_avg_feb_apr,call_avg_may_jul,call_avg_nov_jan,call_std_aug_oct,call_std_feb_apr,call_std_may_jul,call_std_nov_jan
0,6075010100,20.57,18.43,21.29,14.36,8.29,6.96,7.47,7.81
1,6075010200,19.71,18.50,16.36,19.93,9.47,8.21,8.54,7.78
2,6075010300,26.50,26.00,30.14,23.00,11.09,10.38,13.76,7.42
3,6075010400,15.50,12.86,13.50,16.86,7.21,5.26,7.27,7.94
4,6075010500,39.14,42.71,44.86,43.29,12.14,13.67,12.13,13.32


In [11]:
sf_hazards = clean.merge_hazards_and_calls(hazards_sf, call_metrics)
print(f"Final analysis shape: {sf_hazards.shape}")
call_cols = [c for c in sf_hazards.columns if 'call_' in c]
sf_hazards[['census_tract'] + call_cols + clean.HAZARD_COVARIATES].head()

Final analysis shape: (182, 66)


,census_tract,call_avg_aug_oct,call_avg_feb_apr,call_avg_may_jul,call_avg_nov_jan,call_std_aug_oct,call_std_feb_apr,call_std_may_jul,call_std_nov_jan,ozone,...,imp._water_bodies,solid_waste,asthma,low_birth_weight,cardiovascular_disease,education,linguistic_isolation,poverty,unemployment,housing_burden
0,6075023200,27.00,29.14,20.36,27.07,12.84,13.14,9.36,9.87,0.031908,...,11,20.75,123.98,8.09,11.96,25.3,10.4,27.3,6.1,37.0
1,6075023103,15.93,18.07,16.36,18.79,3.45,9.71,6.10,7.30,0.030640,...,14,13.90,123.98,7.74,11.96,21.0,4.5,71.7,6.4,22.3
2,6075023001,15.21,13.79,11.21,17.50,5.66,7.35,6.45,10.62,0.031908,...,10,5.75,123.98,6.41,11.96,27.5,19.8,30.7,4.8,25.6
3,6075023400,20.14,18.57,20.64,20.93,7.58,7.61,8.79,7.81,0.031908,...,11,16.90,123.98,6.29,11.96,29.7,23.1,40.1,6.3,10.4
4,6075023102,25.00,17.86,24.86,26.64,9.91,7.90,7.12,8.12,0.031908,...,14,9.30,123.98,8.70,11.96,17.8,5.5,48.1,11.6,25.5


In [12]:
sf_hazards.info()

<class 'pandas.DataFrame'>
Index: 182 entries, 0 to 190
Data columns (total 66 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   census_tract                 182 non-null    int64  
 1   total_population             182 non-null    int64  
 2   california_county            182 non-null    str    
 3   zip                          182 non-null    int64  
 4   approximate_location         182 non-null    str    
 5   longitude                    182 non-null    float64
 6   latitude                     182 non-null    float64
 7   ces_4.0_score                182 non-null    float64
 8   ces_4.0_percentile           182 non-null    float64
 9   ces_4.0_percentile_range     182 non-null    str    
 10  ozone                        182 non-null    float64
 11  ozone_pctl                   182 non-null    float64
 12  pm2.5                        182 non-null    float64
 13  pm2.5_pctl                   182 non